In [ ]:
!apt-get install -y default-jdk -q

# Download Kafka with a verified mirror
!wget -q "https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz"

# Verify it downloaded
!ls -lh kafka_2.13-3.7.0.tgz

!tar -xzf kafka_2.13-3.7.0.tgz
!mv kafka_2.13-3.7.0 /kafka

# Verify kafka exists
!ls /kafka/bin/

!pip install kafka-python requests -q
print("Done.")

Reading package lists...
Building dependency tree...
Reading state information...
default-jdk is already the newest version (2:1.11-72build2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
-rw-r--r-- 1 root root 114M Feb 26  2024 kafka_2.13-3.7.0.tgz
connect-distributed.sh	      kafka-metadata-quorum.sh
connect-mirror-maker.sh       kafka-metadata-shell.sh
connect-plugin-path.sh	      kafka-mirror-maker.sh
connect-standalone.sh	      kafka-producer-perf-test.sh
kafka-acls.sh		      kafka-reassign-partitions.sh
kafka-broker-api-versions.sh  kafka-replica-verification.sh
kafka-client-metrics.sh       kafka-run-class.sh
kafka-cluster.sh	      kafka-server-start.sh
kafka-configs.sh	      kafka-server-stop.sh
kafka-console-consumer.sh     kafka-storage.sh
kafka-console-producer.sh     kafka-streams-application-reset.sh
kafka-consumer-groups.sh      kafka-topics.sh
kafka-consumer-perf-test.sh   kafka-transactions.sh
kafka-delegation-tokens.sh    kafka-verifiable-consumer.sh


In [ ]:
import subprocess, time

zk = subprocess.Popen(
    ["/kafka/bin/zookeeper-server-start.sh", "/kafka/config/zookeeper.properties"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(6)

broker = subprocess.Popen(
    ["/kafka/bin/kafka-server-start.sh", "/kafka/config/server.properties"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)
print("Zookeeper and Kafka broker running.")

Zookeeper and Kafka broker running.


In [ ]:
import subprocess
result = subprocess.run(
    ["/kafka/bin/kafka-topics.sh", "--create",
     "--topic", "iss-location",
     "--bootstrap-server", "localhost:9092",
     "--partitions", "1",
     "--replication-factor", "1"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Created topic iss-location.




In [ ]:
import threading, requests, time, json
from kafka import KafkaProducer

DURATION = 3600      # change to 3600 for real run

def run_producer():
    producer = KafkaProducer(
        bootstrap_servers="localhost:9092",
        value_serializer=lambda v: json.dumps(v).encode("utf-8")
    )
    start = time.time()
    count = 0
    while time.time() - start < DURATION:
        try:
            r = requests.get("http://api.open-notify.org/iss-now.json", timeout=5).json()
            producer.send("iss-location", r)
            count += 1
            print(f"[Producer {count}] {r['iss_position']}")
        except Exception as e:
            print(f"Producer error: {e}")
        time.sleep(5)
    producer.flush()
    print("Producer done.")

t = threading.Thread(target=run_producer, daemon=True)
t.start()

In [ ]:
from kafka import KafkaConsumer
import json

MAX_POINTS = 720        # change to 720 for real run
TIMEOUT_MS = 3700000     # change to 3700000 for real run

consumer = KafkaConsumer(
    "iss-location",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="earliest",
    consumer_timeout_ms=TIMEOUT_MS
)

readings = []
for msg in consumer:
    data = msg.value
    readings.append({
        "lat": float(data["iss_position"]["latitude"]),
        "lon": float(data["iss_position"]["longitude"]),
        "timestamp": data["timestamp"]
    })
    print(f"[Consumer {len(readings)}] lat={readings[-1]['lat']}, lon={readings[-1]['lon']}")
    if len(readings) >= MAX_POINTS:
        break

print(f"\nTotal points collected: {len(readings)}")

[Consumer 1] lat=8.3977, lon=65.3832
[Consumer 2] lat=8.6498, lon=65.5677
[Consumer 3] lat=8.927, lon=65.7709
[Consumer 4] lat=9.1789, lon=65.9559
[Consumer 5] lat=10.4867, lon=66.9231
[Consumer 6] lat=11.2395, lon=67.4851
[Consumer 7] lat=30.4499, lon=84.4892
[Consumer 8] lat=30.8765, lon=84.9662
[Consumer 9] lat=31.0998, lon=85.2187
[Consumer 10] lat=31.367, lon=85.5234
[Producer 5] {'longitude': '86.0098', 'latitude': '31.7885'}
[Consumer 11] lat=31.7885, lon=86.0098
[Producer 6] {'longitude': '86.2677', 'latitude': '32.0094'}
[Consumer 12] lat=32.0094, lon=86.2677
[Producer 7] {'longitude': '86.5269', 'latitude': '32.2297'}
[Consumer 13] lat=32.2297, lon=86.5269
[Producer 8] {'longitude': '86.8137', 'latitude': '32.4713'}
[Consumer 14] lat=32.4713, lon=86.8137
Producer error: HTTPConnectionPool(host='api.open-notify.org', port=80): Max retries exceeded with url: /iss-now.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x7aca4f5607d0>, 'Connection to

KeyboardInterrupt: 

In [ ]:
import plotly.graph_objects as go

lats = [r["lat"] for r in readings]
lons = [r["lon"] for r in readings]

fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lat=lats, lon=lons,
    mode="lines+markers",
    line=dict(width=1.5, color="cyan"),
    marker=dict(size=3, color="cyan"),
    name="ISS Path"
))

fig.add_trace(go.Scattergeo(
    lat=[lats[-1]], lon=[lons[-1]],
    mode="markers",
    marker=dict(size=12, color="red", symbol="star"),
    name="Last Position"
))

fig.update_layout(
    title="ISS Satellite Tracking — 1 Hour (Kafka)",
    geo=dict(
        showland=True, landcolor="lightgray",
        showocean=True, oceancolor="lightblue",
        showcoastlines=True, coastlinecolor="white",
        projection_type="natural earth",
        bgcolor="black"
    ),
    paper_bgcolor="black",
    font_color="white"
)

fig.show()